# Nuclear scaling — analysis

One run of the nuclear-scaling pipeline, analysed as time series. Four tracked
nuclei (`FOCAL`) are followed through every analysis so individual trajectories
stay visible next to the population summary — the question is how a nucleus
changes, not what the population looked like once.

`Nucleus_ID` in the export is the **track id**, so a nucleus keeps its identity
across frames and `FOCAL` selects the longest-lived tracks.

### Data source

Set `DATA_SOURCE` in the setup cell. Switching needs a kernel restart.

| `DATA_SOURCE` | Reads | Where |
|---|---|---|
| `"csv"` | `<RUN_DIR>/database_export/*.csv` | Cheaha — no SQLite DB there |
| `"db"` | `nuclear_scaling.db` + parquet sidecars via `src/nsdb.py` | StarForge — `starforge` kernel (needs pyarrow) |

Both backends hand the rest of the notebook the same snake_case tables, so
nothing after **2 · Load data** knows where the data came from. All plotting
code lives in these cells so you can edit it directly.

### Layout

| § | Contents |
|---|---|
| 1 | Setup — paths, backend, **every tunable constant** |
| 2 | Load data — the only cell that touches CSVs or the DB |
| 3 | Inventory and integrity |
| 4 | Shared analysis helpers |
| 5–15 | Analysis and figures |
| 16 | Figures written |
| A | Diagnostics (one-off) |

### Known limits of the v18.1 export

| Limitation | Consequence |
|---|---|
| `Background_*`, `Halo*_NPC`, `Halo*_Membrane` empty | `nc_ratio` has no background subtraction — read the trend, not the level |
| `Rho_Normalized` is centre→wall | radial position is in ρ, not µm |
| `Distance_px` not exported | no absolute radii anywhere |
| sweep runs with `exclude_nucleus=True` | nothing inside the nucleus mask; `rho_wall` (0 = mask surface, 1 = wall) is exact w.r.t. the mask, from each ray's minimum ρ |
| repair not written back to `grouped_z_df` | `nucleus_z_stack` areas are pre-repair — use `nuclei` for area |
| `Droplet_ID` empty | droplet unavailable as a grouping level |

## 1 · Setup

In [ ]:
from pathlib import Path
import os, sys, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)


# ---------------------------------------------------------------------------------
# Data source. PERMANENT config (2026-09-16 refactor), not a debug switch.
#   "csv" -> the run's database_export/ CSVs    (Cheaha: the SQLite DB isn't there)
#   "db"  -> nuclear_scaling.db via src/nsdb.py (StarForge, starforge kernel)
# ---------------------------------------------------------------------------------
DATA_SOURCE = "csv"

# Run ID is <experiment>__cfg-<hash>; EXPERIMENT is parsed from it so the two can't drift.
# On the "db" backend only the name is used, so the directory needn't exist there.
RUN_DIR = Path("/data/user/tdeibert/Nuclear_Scaling/Runs/control_extract_1.1__cfg-ea0f577d")

# ---- analysis constants ---------------------------------------------------------
N_FOCAL  = 4        # longest-lived tracks, followed through every figure
BIN_MIN  = 6.0      # acquisition-time bin width (min)
RHO_BINS = 60       # radial bins for theta x rho grids and surfaces
GAP      = 0.9      # perinuclear rose: fraction of the surface-to-wall gap included
BAND     = 0.15     # envelope band: fraction of the surface-to-wall gap treated as envelope
RNG_SEED = 0        # boxplot jitter, so saved figures are identical between runs

# Membrane-asymmetry shells. rho_wall is referenced to the nucleus mask surface
# (0 = surface, 1 = droplet wall). For centre-referenced shells instead, set
#   ASYM_RADIUS = "rho_normalized"
#   ASYM_SHELLS = [(0.00, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 1.00)]
ASYM_RADIUS      = "rho_wall"
ASYM_SHELLS      = [(0.00, 0.10), (0.10, 0.25), (0.25, 0.50), (0.50, 1.00)]
RING_SHELL_INDEX = 1    # position in ASYM_SHELLS of the shell holding the ring (§13a, §13b)

SHELL_COLORS = ["#7fd4b0", "#2fa383", "#4c6fbf", "#1a1a2e"]
FOCAL_COLORS = ["#c1121f", "#0a6f8a", "#e08214", "#5b3a91"]


# ---- validate config (PERMANENT guards) -----------------------------------------
if DATA_SOURCE not in ("csv", "db"):
    raise ValueError(f"DATA_SOURCE must be 'csv' or 'db', got {DATA_SOURCE!r}")
if ASYM_RADIUS not in ("rho_wall", "rho_normalized"):
    raise ValueError(f"ASYM_RADIUS must be 'rho_wall' or 'rho_normalized', got {ASYM_RADIUS!r}")
if not 0 <= RING_SHELL_INDEX < len(ASYM_SHELLS):
    raise ValueError(f"RING_SHELL_INDEX {RING_SHELL_INDEX} outside ASYM_SHELLS")
if len(SHELL_COLORS) < len(ASYM_SHELLS) or len(FOCAL_COLORS) < N_FOCAL:
    raise ValueError("need at least one colour per shell and per focal track")

# One backend per kernel: switching would leave the other backend's tables and module
# in memory, and cells would silently mix the two.
if globals().get("_ACTIVE_SOURCE", DATA_SOURCE) != DATA_SOURCE:
    raise RuntimeError(f"kernel already loaded DATA_SOURCE={_ACTIVE_SOURCE!r} -- "
                       f"restart the kernel before switching to {DATA_SOURCE!r}")
_ACTIVE_SOURCE = DATA_SOURCE

RUN_ID = RUN_DIR.name
EXPERIMENT, _sep, CFG_HASH = RUN_ID.partition("__cfg-")
if not _sep:
    raise ValueError(f"run dir name doesn't match '<experiment>__cfg-<hash>': {RUN_ID}")


# ---- backend paths ---------------------------------------------------------------
if DATA_SOURCE == "csv":
    EXPORT_DIR = RUN_DIR / "database_export"
    FIGDIR     = RUN_DIR / "figures"
    for p in (RUN_DIR, EXPORT_DIR):
        if not p.is_dir():
            raise FileNotFoundError(f"not found: {p}")
    SOURCE_DESC = str(EXPORT_DIR)

else:
    def find_project_root(marker="src/nsdb.py", start=None):
        """Anchor on a file so this runs from any machine or subdirectory."""
        here = (start or Path.cwd()).resolve()
        for d in (here, *here.parents):
            if (d / marker).exists():
                return d
        raise FileNotFoundError(f"no {marker} in {here} or any parent")

    PROJECT = find_project_root()
    DB      = PROJECT / "data" / "db" / "nuclear_scaling.db"
    FIGDIR  = PROJECT / "outputs" / "figures"
    if not DB.is_file():
        raise FileNotFoundError(f"database not found: {DB}")
    os.environ["NUCLEAR_SCALING_DB"]   = str(DB)
    os.environ["NUCLEAR_SCALING_ROOT"] = str(PROJECT / "data" / "derived")
    if str(PROJECT / "src") not in sys.path:
        sys.path.insert(0, str(PROJECT / "src"))
    import nsdb
    SOURCE_DESC = f"{DB} ({DB.stat().st_size/1e6:.1f} MB)"

FIGDIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60, "display.width", 180)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10})

print("python    :", sys.executable)
print("source    :", DATA_SOURCE, "->", SOURCE_DESC)
print("experiment:", EXPERIMENT, "| cfg:", CFG_HASH)
print("figures   :", FIGDIR)

## 2 · Load data

The only cell that knows about CSVs or the DB. Both backends produce:

| Name | Contents |
|---|---|
| `experimental_cfg` | experiment metadata |
| `df_all` / `df` | nuclei for `EXPERIMENT` — all QC / `QC_Flag == PASS` |
| `load_sweep(frames)` | → `(sweep, channel)`; the radial profile, loaded in §9 because it is millions of rows |

CSV headers are lower-cased to match the DB's snake_case names. A required column
that isn't there raises here, listing the columns that are, rather than failing
inside a plot.

In [ ]:
# PERMANENT data-access layer (2026-09-16 refactor), replacing the ad-hoc CSV loader,
# the get_nuclei() port and the direct nsdb calls that used to be spread through the notebook.
NUCLEI_REQUIRED = ["nucleus_id", "time_frame", "time_min", "cross_sectional_area_um2", "nc_ratio"]
RADIAL_REQUIRED = ["nucleus_id", "time_frame", "theta_deg", "rho_normalized", "channel", "intensity"]
QC_LABELS       = {"PASS", "FAIL"}


def require_columns(table, required, what):
    missing = [c for c in required if c not in table.columns]
    if missing:
        raise KeyError(f"{what}: missing {missing}\navailable: {list(table.columns)}")


if DATA_SOURCE == "csv":
    EXPORT_STEMS = ["Experimental_Cfg", "Nuclei", "NucleusZStack", "RadialProfile", "RawIntensities"]

    def export_csv(stem):
        path = EXPORT_DIR / f"{stem}.csv"
        if not path.is_file():
            raise FileNotFoundError(f"missing from export: {path}")
        return path

    def export_header(stem):
        """Lower-cased column name -> column name as written in <stem>.csv (header only)."""
        cols = pd.read_csv(export_csv(stem), nrows=0).columns
        lower = {c.lower(): c for c in cols}
        if len(lower) != len(cols):
            raise ValueError(f"{stem}.csv: column names collide when lower-cased: {list(cols)}")
        return lower

    def read_export(stem, columns=None):
        """Read <stem>.csv with snake_case column names; `columns` limits what is parsed."""
        header = export_header(stem)
        if columns is not None:
            missing = [c for c in columns if c not in header]
            if missing:
                raise KeyError(f"{stem}.csv: missing {missing}\navailable: {list(header)}")
        usecols = None if columns is None else [header[c] for c in columns]
        out = pd.read_csv(export_csv(stem), usecols=usecols, low_memory=False)
        out.columns = out.columns.str.lower()
        return out if columns is None else out[columns]

    def single_experiment(table, what):
        """The run directory holds exactly one experiment; anything else is the wrong export."""
        present = set(table["experiment_id"].astype(str).unique())
        if present != {EXPERIMENT}:
            raise ValueError(f"{what} holds experiments {sorted(present)}, expected only {EXPERIMENT!r}")

    extras = sorted(p.name for p in EXPORT_DIR.iterdir()
                    if p.suffix != ".csv" or p.stem not in EXPORT_STEMS)
    if extras:
        print("!! unexpected entries in export (not loaded):", extras)

    experimental_cfg = read_export("Experimental_Cfg")
    nucleus_z_stack  = read_export("NucleusZStack")
    raw_intensities  = read_export("RawIntensities")
    nuclei_export    = read_export("Nuclei")

    require_columns(nuclei_export, NUCLEI_REQUIRED + ["experiment_id", "qc_flag"], "Nuclei.csv")
    single_experiment(nuclei_export, "Nuclei.csv")

    qc = nuclei_export["qc_flag"].astype(object).where(nuclei_export["qc_flag"].notna())
    qc = qc.map(lambda v: v if pd.isna(v) else str(v).strip().upper())
    bad = sorted(set(qc.dropna()) - QC_LABELS)
    if bad or qc.isna().any():
        raise ValueError(f"QC_Flag: unexpected labels {bad}, {int(qc.isna().sum())} missing "
                         f"-- decide how they should be treated before filtering on PASS")

    df_all = nuclei_export.assign(qc_flag=qc).reset_index(drop=True)
    df     = df_all[df_all["qc_flag"] == "PASS"].reset_index(drop=True)

    def load_sweep(frames):
        """Radial profile for `frames`, single channel -> (sweep, channel)."""
        has_exp = "experiment_id" in export_header("RadialProfile")
        sweep = read_export("RadialProfile",
                            columns=RADIAL_REQUIRED + (["experiment_id"] if has_exp else []))
        if has_exp:
            single_experiment(sweep, "RadialProfile.csv")
            sweep = sweep.drop(columns="experiment_id")
        channels = sweep["channel"].dropna().unique()
        if len(channels) != 1:
            raise ValueError(f"expected one radial channel, got {list(channels)}")
        keep = (sweep["channel"] == channels[0]) & sweep["time_frame"].isin(list(frames))
        return sweep[keep].reset_index(drop=True), channels[0]

else:
    experimental_cfg = nsdb.experiments()
    nucleus_z_stack = raw_intensities = None     # not used by the analysis; nsdb.audit() covers integrity
    df_all = nsdb.nuclei(experiment=EXPERIMENT, qc=None).reset_index(drop=True)
    df     = nsdb.nuclei(experiment=EXPERIMENT, qc="PASS").reset_index(drop=True)

    def load_sweep(frames):
        """Radial profile for `frames`, single channel -> (sweep, channel). Filters in the parquet reader."""
        channels = nsdb.radial(EXPERIMENT, columns=["channel"])["channel"].dropna().unique()
        if len(channels) != 1:
            raise ValueError(f"expected one radial channel, got {list(channels)}")
        sweep = nsdb.radial(EXPERIMENT, columns=RADIAL_REQUIRED,
                            filters=[("channel", "==", channels[0]),
                                     ("time_frame", "in", list(frames))])
        return sweep.reset_index(drop=True), channels[0]


require_columns(df_all, NUCLEI_REQUIRED, f"nuclei ({DATA_SOURCE})")
if df.empty:
    raise ValueError(f"no PASS nuclei for {EXPERIMENT!r}")

print(f"nuclei: {len(df_all):,} rows, {df_all.nucleus_id.nunique():,} tracks | "
      f"PASS {len(df):,} / FAIL {len(df_all) - len(df):,}")

## 3 · Inventory and integrity

On `"db"` this is `nsdb.audit()`. On `"csv"` the checks are a CSV-level stand-in
written for this notebook — **not** a port of `nsdb.audit()`, which may check
more. `skip` means a column the check needs isn't in the export. Nothing here
stops the notebook.

In [ ]:
# PERMANENT, informational: inventory + integrity report. Does not raise.
EXP_COLS = ["experiment_id", "experimental_group", "treatment",
            "concentration", "biological_replicate", "experiment_date"]
display(experimental_cfg[[c for c in EXP_COLS if c in experimental_cfg.columns]])

if DATA_SOURCE == "db":
    report = nsdb.audit()
    display(report)
    issues = report.loc[~report.ok, "check"].tolist()

else:
    checks = []
    def check(name, ok, detail=""):
        status = "skip" if ok is None else ("ok" if ok else "FAIL")
        checks.append(dict(check=name, status=status, detail=detail))

    check("experimental_cfg: exactly one row", len(experimental_cfg) == 1,
          f"{len(experimental_cfg)} rows")

    n_dup = int(df_all.duplicated(["nucleus_id", "time_frame"]).sum())
    check("nuclei: one row per (nucleus_id, time_frame)", n_dup == 0, f"{n_dup} duplicates")

    for c in ["time_min", "cross_sectional_area_um2", "nc_ratio"]:
        n = int(df[c].isna().sum())
        check(f"nuclei PASS: no missing {c}", n == 0, f"{n} missing")

    med_t = df_all.groupby("time_frame")["time_min"].median()
    check("nuclei: median time_min increases with time_frame", med_t.is_monotonic_increasing,
          f"{len(med_t)} frames")

    check("raw_intensities: one row per nucleus row", len(raw_intensities) == len(df_all),
          f"{len(raw_intensities):,} vs {len(df_all):,}")

    if "nucleus_id" in nucleus_z_stack.columns:
        orphans = set(nucleus_z_stack["nucleus_id"]) - set(df_all["nucleus_id"])
        check("nucleus_z_stack: every nucleus_id is in nuclei", not orphans,
              f"{len(orphans)} orphan ids")
    else:
        check("nucleus_z_stack: every nucleus_id is in nuclei", None, "no nucleus_id column")

    report = pd.DataFrame(checks)
    display(report)
    issues = report.loc[report.status == "FAIL", "check"].tolist()

print("open issues:", *(issues or ["none"]), sep="\n  ")

## 4 · Shared analysis helpers

The only reusable code in the notebook. Everything below is plotting you can
edit in place.

**Angle convention.** The sweep casts rays as `y = cy + r·sin θ` and image y
increases downward, so θ runs *clockwise* on screen with θ=0 pointing right.
`orient_polar` matches that.

In [ ]:
# ---- plotting utilities ----------------------------------------------------------
def orient_polar(ax):
    """theta=0 at +x, increasing clockwise — matches the displayed image."""
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(-1)


def boxplot(ax, data, labels, **kw):
    """boxplot with whichever tick-label kwarg this matplotlib accepts.

    `labels=` became `tick_labels=` in matplotlib 3.9 and was later removed.
    """
    try:
        return ax.boxplot(data, tick_labels=labels, **kw)
    except TypeError:
        return ax.boxplot(data, labels=labels, **kw)


def short(nid):
    """'control_extract_1.1|FOV01|N000082' -> 'N000082'."""
    return str(nid).split("|")[-1]


def median_iqr(d, col, by="time_frame", t="time_min"):
    """Per-group median and IQR of `col`, with the group's median acquisition time as x."""
    g = d.groupby(by)
    return pd.DataFrame({"t": g[t].median(), "m": g[col].median(),
                         "lo": g[col].quantile(.25), "hi": g[col].quantile(.75)})


def plot_population(ax, d, col, points=True, label="population median"):
    """Grey points, black median line and IQR band per time frame."""
    if points:
        ax.scatter(d["time_min"], d[col], s=6, alpha=.15, color="0.5", lw=0)
    q = median_iqr(d, col)
    ax.plot(q.t, q.m, "-", color="k", lw=2, label=label)
    ax.fill_between(q.t, q.lo, q.hi, color="k", alpha=.10, lw=0)


def plot_tracks(ax, d, col, colors, x="time_min"):
    """One coloured line per track; `colors` maps nucleus_id -> colour."""
    for nid, color in colors.items():
        g = d[d["nucleus_id"] == nid].sort_values(x)
        ax.plot(g[x], g[col], "-o", ms=5, lw=1.6, color=color, label=short(nid))


def time_bins(t, width=BIN_MIN):
    """Fixed-width acquisition-time bins -> (edges, labels 'lo-hi')."""
    edges = np.arange(0, np.nanmax(t) + width, width)
    return edges, [f"{int(x)}-{int(y)}" for x, y in zip(edges[:-1], edges[1:])]


# ---- radial geometry -------------------------------------------------------------
def add_rho_wall(sweep):
    """Surface-referenced radius per ray: 0 at the nucleus mask surface, 1 at the wall.

    The sweep runs with exclude_nucleus=True, so each ray's samples start at the
    first pixel outside the mask and the ray's MINIMUM rho is its boundary crossing:

        rho_wall = (rho - rho_surface) / (1 - rho_surface)

    Exact with respect to the mask — and the masks are what Vulcan 2.0 is meant to
    fix, so an under-segmented nucleus puts "envelope" samples inside the real envelope.
    """
    key = ["nucleus_id", "time_frame", "theta_deg"]
    rho_surface = sweep.groupby(key)["rho_normalized"].transform("min")
    rho_wall = ((sweep["rho_normalized"] - rho_surface)
                / (1 - rho_surface).replace(0, np.nan)).clip(0, 1)
    return sweep.assign(rho_surface=rho_surface, rho_wall=rho_wall)


def ray_peaks(sweep):
    """Per ray: the rho at which intensity peaks (the pipeline plots this in µm)."""
    idx = sweep.groupby(["nucleus_id", "time_frame", "theta_deg"])["intensity"].idxmax()
    return (sweep.loc[idx, ["nucleus_id", "time_frame", "theta_deg",
                            "rho_normalized", "intensity"]]
            .rename(columns={"rho_normalized": "rho_at_peak",
                             "intensity": "peak_intensity"}))


def angular_profile(d, n_bins=36):
    """Bin one nucleus/frame/shell by angle -> (theta_rad, mean_intensity, n_samples).

    NaN where no ray sampled that wedge — that happens where the droplet wall
    clips the sweep, which is exactly why R_geom below is needed.
    """
    edges = np.linspace(0, 360, n_bins + 1)
    idx = pd.cut(d["theta_deg"] % 360, edges, labels=False, include_lowest=True)
    g = d.assign(_b=idx).groupby("_b")["intensity"]
    mean = g.mean().reindex(range(n_bins)).to_numpy(dtype=float)
    count = g.size().reindex(range(n_bins)).fillna(0).to_numpy(dtype=float)
    return np.radians(0.5 * (edges[:-1] + edges[1:])), mean, count


def sweep_grid(d, rho_bins=RHO_BINS, radius_col="rho_normalized", agg="mean", close_theta=True):
    """Bin a long-form sweep into a regular theta x rho grid for plot_surface.

    Each ray has a different sample count, so rho must be binned before the
    pivot. theta is closed by repeating the first ray at +360 so there is no seam.
    """
    edges = np.linspace(0, 1, rho_bins + 1)
    d = d[d[radius_col].between(0, 1)].copy()
    d["rb"] = pd.cut(d[radius_col], edges, labels=False, include_lowest=True)
    grid = (d.pivot_table(index="theta_deg", columns="rb", values="intensity", aggfunc=agg)
            .reindex(columns=range(rho_bins)))
    theta, z = grid.index.to_numpy(dtype=float), grid.to_numpy(dtype=float)
    if close_theta and theta.size > 1:
        theta = np.append(theta, theta[0] + 360.0)
        z = np.vstack([z, z[0]])
    rho = 0.5 * (edges[:-1] + edges[1:])
    THETA, RHO = np.meshgrid(theta, rho, indexing="ij")
    return THETA, RHO, z


def fill_gaps(z):
    """Interpolate NaN holes along rho. plot_surface renders NaN as a hole, and
    outer rho bins go sparse as rays shorten. Does not extrapolate past the
    ends — a bin with no data anywhere stays a hole, because that is real."""
    out = z.copy()
    for i in range(out.shape[0]):
        row, ok = out[i], ~np.isnan(out[i])
        if ok.sum() >= 2:
            idx = np.arange(row.size)
            out[i] = np.interp(idx, idx[ok], row[ok], left=np.nan, right=np.nan)
    return out


def polar_xy(THETA, RHO):
    """theta/rho grid -> Cartesian, so a surface sits over the nucleus footprint."""
    th = np.radians(THETA)
    return RHO * np.cos(th), RHO * np.sin(th)


# ---- membrane asymmetry ----------------------------------------------------------
def shell_label(lo, hi):
    return f"({lo:.2f}, {hi:.2f}]"


def resultant(theta_rad, weight):
    """Weighted circular resultant -> (R, phi). R=0 isotropic, R=1 one direction."""
    w = np.asarray(weight, dtype=float)
    ok = np.isfinite(w) & np.isfinite(theta_rad) & (w > 0)
    if ok.sum() == 0 or w[ok].sum() <= 0:
        return np.nan, np.nan
    z = np.sum(w[ok] * np.exp(1j * theta_rad[ok])) / w[ok].sum()
    return float(abs(z)), float(np.angle(z))


def asymmetry(sweep, shells, radius_col, n_bins=36, baseline_pct=10.0):
    """Per (nucleus, frame, shell) membrane asymmetry, shells taken on `radius_col`.

    R       intensity-weighted resultant after subtracting a per-shell baseline
    phi     its direction (radians, image convention)
    R_geom  resultant of the SAMPLE COUNTS alone — the apparent asymmetry an
            isotropic nucleus would show from droplet-wall clipping. R has to
            beat R_geom to mean anything.
    p       Rayleigh approximation exp(-n_eff*R^2), where n_eff is the effective
            weight count, so a profile carried by two bright bins is not
            credited with the full bin count.
    """
    rows = []
    for (nid, t), g in sweep.groupby(["nucleus_id", "time_frame"], sort=False):
        for lo, hi in shells:
            sub = g[(g[radius_col] > lo) & (g[radius_col] <= hi)]
            if sub.empty:
                continue
            th, mean, count = angular_profile(sub, n_bins)
            filled = np.isfinite(mean)
            if filled.sum() < 3:
                continue
            w = np.clip(mean - np.nanpercentile(mean[filled], baseline_pct), 0, None)
            w[~filled] = 0.0
            R, phi = resultant(th, w)
            Rg, _ = resultant(th, count)
            n_eff = (w.sum() ** 2 / np.sum(w ** 2)) if np.sum(w ** 2) > 0 else 0.0
            rows.append(dict(nucleus_id=nid, time_frame=t, shell=shell_label(lo, hi),
                             shell_lo=lo, shell_hi=hi, R=R, phi=phi, R_geom=Rg,
                             n_eff=n_eff, n_samples=len(sub),
                             p=float(np.exp(-n_eff * R ** 2)) if R == R else np.nan))
    return pd.DataFrame(rows)


def population_direction(asym):
    """Do nuclei agree on a lab-frame direction? Each nucleus gets unit weight."""
    rows = []
    for (shell, t), g in asym.groupby(["shell", "time_frame"]):
        phi = g["phi"].dropna().to_numpy()
        if phi.size == 0:
            continue
        R, mean_phi = resultant(phi, np.ones_like(phi))
        rows.append(dict(shell=shell, time_frame=t, n=phi.size, R_pop=R,
                         mean_phi=mean_phi, p=float(np.exp(-phi.size * R ** 2))))
    return pd.DataFrame(rows)


print("helpers loaded")

## 5 · Focal tracks

`FOCAL` is the `N_FOCAL` longest-lived tracks — the ones with enough timepoints
to show a trajectory rather than a point. `FRAMES` are the frames they span.

In [ ]:
span = (df_all.groupby("nucleus_id")
        .agg(n_frames=("time_frame", "nunique"),
             first=("time_frame", "min"), last=("time_frame", "max"))
        .sort_values(["n_frames", "first"], ascending=[False, True]))
FOCAL = span.head(N_FOCAL).index.tolist()
FC    = dict(zip(FOCAL, FOCAL_COLORS))
if len(FOCAL) < N_FOCAL:
    print(f"!! only {len(FOCAL)} tracks available (N_FOCAL={N_FOCAL})")

FRAMES = sorted(df_all.loc[df_all.nucleus_id.isin(FOCAL), "time_frame"].unique().tolist())   # .tolist(): plain ints in titles/prints
_cmap = plt.get_cmap("viridis")
FRAME_COLORS = {t: _cmap(i / max(len(FRAMES) - 1, 1)) for i, t in enumerate(FRAMES)}

print("focal tracks:")
display(span.head(N_FOCAL))
print("frames:", FRAMES)

## 6 · Cross-sectional area over time

Population in grey, the focal tracks in colour. `time_min` is the true per-tile
acquisition time, not `time_frame x 6` — nuclei in different tiles of one frame
were imaged up to 5 minutes apart.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(14, 5))

plot_population(a, df, "cross_sectional_area_um2")
plot_tracks(a, df_all, "cross_sectional_area_um2", FC)
a.set_xlabel("true acquisition time (min)")
a.set_ylabel("cross-sectional area (um^2)")
a.set_title("(a) Area over time — population and focal tracks")
a.legend(frameon=False, fontsize=8)

for nid in FOCAL:
    g = df_all[df_all.nucleus_id == nid].sort_values("time_min")
    if g.empty:
        continue
    b.plot(g.time_min, g.cross_sectional_area_um2 / g.cross_sectional_area_um2.iloc[0],
           "-o", ms=5, lw=1.6, color=FC[nid], label=short(nid))
b.axhline(1.0, color="k", lw=.7, ls="--")
b.set_xlabel("true acquisition time (min)")
b.set_ylabel("area / area at first frame")
b.set_title("(b) Fold change within each track")
b.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(FIGDIR / "area_timecourse.png", bbox_inches="tight")

### 6b · Binned distribution with the focal tracks overlaid

In [ ]:
rng = np.random.default_rng(RNG_SEED)      # local, so re-running the cell gives the same jitter
edges, labels = time_bins(df.time_min)
d = df.assign(bin=pd.cut(df.time_min, edges, labels=labels,
                         include_lowest=True)).dropna(subset=["bin"])
order = [c for c in labels if (d["bin"] == c).any()]
pos = {c: i + 1 for i, c in enumerate(order)}
data = [d.loc[d["bin"] == c, "cross_sectional_area_um2"].to_numpy() for c in order]

fig, ax = plt.subplots(figsize=(12, 5.5))
bp = boxplot(ax, data, order, patch_artist=True, showfliers=False,
             medianprops=dict(color="k"))
for box in bp["boxes"]:
    box.set(facecolor="#b8c6e8", alpha=.75, edgecolor="#33415c")
for i, v in enumerate(data, start=1):
    ax.scatter(rng.normal(i, .07, v.size), v, s=4, color="k", alpha=.18, lw=0)

for nid in FOCAL:
    g = df_all[df_all.nucleus_id == nid].copy()
    g["bin"] = pd.cut(g.time_min, edges, labels=labels, include_lowest=True).astype(str)
    g = g[g["bin"].isin(pos)].sort_values("time_min")
    ax.plot(g["bin"].map(pos), g.cross_sectional_area_um2,
            "-o", ms=5, lw=1.6, color=FC[nid], label=short(nid), zorder=5)

# n= labels last, so they sit above the final y-limit rather than one set before the tracks
for i, v in enumerate(data, start=1):
    ax.annotate(f"n={v.size}", (i, ax.get_ylim()[1]), ha="center", fontsize=8,
                color="0.35", xytext=(0, 4), textcoords="offset points")

ax.set_xlabel(f"acquisition time bin (min, width {int(BIN_MIN)})")
ax.set_ylabel("cross-sectional area (um^2)")
ax.set_title("Nuclear cross-sectional area per time interval, with focal tracks", pad=18)
ax.legend(frameon=False, fontsize=8)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.savefig(FIGDIR / "area_bins.png", bbox_inches="tight")

## 7 · N/C ratio over time

From the mCherry halos with **no background subtraction** — the `background_*`
columns are empty in the export. Read the shape, not the absolute level.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_population(ax, df, "nc_ratio")
plot_tracks(ax, df_all, "nc_ratio", FC)
ax.set_xlabel("true acquisition time (min)")
ax.set_ylabel("N/C ratio")
ax.set_title("N/C ratio over time  (no background subtraction)")
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIGDIR / "nc_ratio.png", bbox_inches="tight")

## 8 · N/C ratio and area, dual axis

One panel per focal track, so the two quantities can be read against each other
within a single nucleus. The population panel comes first for reference.

In [ ]:
def dual(ax, t, nc, area, title, nc_band=None, area_band=None):
    """N/C on the left axis, area on the right, sharing a time axis."""
    ax.plot(t, nc, "-o", color="#14746f", ms=4, label="N/C ratio")
    if nc_band is not None:
        ax.fill_between(t, *nc_band, color="#14746f", alpha=.15, lw=0)
    ax.set_ylabel("N/C ratio", color="#14746f")
    ax.tick_params(axis="y", labelcolor="#14746f")
    ax.set_xlabel("time (min)")
    ax2 = ax.twinx()
    ax2.plot(t, area, "-s", color="#c1121f", ms=4, label="area")
    if area_band is not None:
        ax2.fill_between(t, *area_band, color="#c1121f", alpha=.13, lw=0)
    ax2.set_ylabel("area (um^2)", color="#c1121f")
    ax2.tick_params(axis="y", labelcolor="#c1121f")
    ax2.spines["right"].set_visible(True)
    ax.set_title(title, fontsize=10)
    return ax2


fig, axes = plt.subplots(1, len(FOCAL) + 1, figsize=(4.6 * (len(FOCAL) + 1), 4.4), squeeze=False)
axes = axes[0]

q_nc, q_area = median_iqr(df, "nc_ratio"), median_iqr(df, "cross_sectional_area_um2")
ax2 = dual(axes[0], q_nc.t, q_nc.m, q_area.m, "population (medians +/- IQR)",
           nc_band=(q_nc.lo, q_nc.hi), area_band=(q_area.lo, q_area.hi))
h1, l1 = axes[0].get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
axes[0].legend(h1 + h2, l1 + l2, frameon=False, fontsize=8, loc="lower right")

for ax, nid in zip(axes[1:], FOCAL):
    g = df_all[df_all.nucleus_id == nid].sort_values("time_min")
    dual(ax, g.time_min, g.nc_ratio, g.cross_sectional_area_um2, short(nid))

fig.suptitle("N/C ratio (teal, left) and cross-sectional area (red, right)")
fig.tight_layout(rect=[0, 0, 1, .93])
fig.savefig(FIGDIR / "dual_axis.png", bbox_inches="tight")

## 9 · Load the radial sweep

Millions of rows, so only the frames the focal tracks span and only the columns
used are loaded. The sweep ran on a single channel, detected rather than assumed.
`rho_wall` is added here once (see `add_rho_wall`) and used by every later section.

In [ ]:
sweep, CHANNEL = load_sweep(FRAMES)
require_columns(sweep, RADIAL_REQUIRED, f"radial sweep ({DATA_SOURCE})")
sweep = add_rho_wall(sweep)

missing = [n for n in FOCAL if n not in set(sweep.nucleus_id)]
if missing:
    print("!! focal tracks absent from the sweep:", [short(m) for m in missing])
orphans = set(sweep.nucleus_id) - set(df_all.nucleus_id)
if orphans:
    print(f"!! {len(orphans)} sweep nucleus_ids not in the nuclei table")

n_rays = sweep.groupby(["nucleus_id", "time_frame", "theta_deg"]).ngroups
print(f"channel: {CHANNEL} | {len(sweep):,} samples, {n_rays:,} rays, "
      f"{sweep.nucleus_id.nunique():,} nuclei | {sweep.memory_usage(deep=True).sum()/1e6:.0f} MB")
sweep.head()

## 10 · Perinuclear rose — focal tracks across time

Rows are the focal tracks, columns timepoints. Wedges are mean intensity in each
angular bin between the nucleus mask surface and `GAP` of the way to the droplet
wall; colour tracks the same value. A lobe that persists across a row is a stable
polarity; one that moves is not.

In [ ]:
NBINS = 24
band = sweep[(sweep.rho_wall > 0) & (sweep.rho_wall <= GAP)]

profiles, vmax = {}, 0.0
for nid in FOCAL:
    for t in FRAMES:
        sub = band[(band.nucleus_id == nid) & (band.time_frame == t)]
        if sub.empty:
            continue
        th, mean, _ = angular_profile(sub, NBINS)
        profiles[(nid, t)] = (th, np.nan_to_num(mean))
        vmax = max(vmax, float(np.nanmax(mean)))

fig, axes = plt.subplots(len(FOCAL), len(FRAMES),
                         figsize=(2.5 * len(FRAMES), 2.7 * len(FOCAL)),
                         subplot_kw={"projection": "polar"}, squeeze=False)
width = 2 * np.pi / NBINS
norm = plt.Normalize(0, vmax)
for r, nid in enumerate(FOCAL):
    for k, t in enumerate(FRAMES):
        ax = axes[r][k]; orient_polar(ax)
        ax.set_xticklabels([]); ax.set_yticklabels([])
        ax.set_ylim(0, vmax * 1.05)
        if (nid, t) not in profiles:
            ax.set_facecolor("0.97")
        else:
            th, vals = profiles[(nid, t)]
            ax.bar(th - width / 2, vals, width=width, align="edge",
                   color=plt.get_cmap("magma")(norm(vals)), edgecolor="none")
        if r == 0:
            ax.text(.5, 1.22, f"t = {t}", transform=ax.transAxes, ha="center", fontsize=9)
    axes[r][0].set_ylabel(short(nid), fontsize=9, labelpad=26)

fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="magma"), ax=axes,
             shrink=.5, label=f"mean {str(CHANNEL).lower()} intensity")
fig.suptitle(f"Perinuclear rose — mask surface to {GAP} of the surface-to-wall gap\n"
             "blank panels are frames where the track was not detected", fontsize=12)
fig.savefig(FIGDIR / "rose_perinuclear.png", bbox_inches="tight")

## 11 · Envelope rose — radius of peak intensity by angle

Focal tracks, all timepoints overlaid by colour. The pipeline plots this in µm;
only ρ survives the export, so the shape is comparable and the scale is not.

In [ ]:
peaks = ray_peaks(sweep)

fig, axes = plt.subplots(1, len(FOCAL), figsize=(4.3 * len(FOCAL), 4.8),
                         subplot_kw={"projection": "polar"}, squeeze=False)
for ax, nid in zip(axes[0], FOCAL):
    orient_polar(ax)
    for t, g in peaks[peaks.nucleus_id == nid].groupby("time_frame"):
        g = g.sort_values("theta_deg")
        th, r = np.radians(g.theta_deg.to_numpy()), g.rho_at_peak.to_numpy()
        ax.plot(np.append(th, th[0]), np.append(r, r[0]), lw=.9, color=FRAME_COLORS[t])
    ax.set_title(short(nid), fontsize=10)
    ax.set_ylim(0, 1)

fig.legend(handles=[plt.Line2D([], [], color=FRAME_COLORS[t], label=f"t={t}") for t in FRAMES],
           loc="center right", frameon=False, fontsize=8, title="frame")
fig.suptitle("Radius of peak intensity by angle (rho, centre->wall) — focal tracks over time")
fig.tight_layout(rect=[0, 0, .93, .93])
fig.savefig(FIGDIR / "rose_individual.png", bbox_inches="tight")

### 11b · Pooled over all nuclei, per frame

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw={"projection": "polar"})
orient_polar(ax)

for t in FRAMES:
    g = (peaks[peaks.time_frame == t].groupby("theta_deg")["rho_at_peak"]
         .agg(med="median", q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75))
         .reset_index().sort_values("theta_deg"))
    if g.empty:
        continue
    th = np.radians(g.theta_deg.to_numpy()); thc = np.append(th, th[0])
    ax.plot(thc, np.append(g["med"], g["med"].iloc[0]), lw=1.1, color=FRAME_COLORS[t], label=f"t={t}")
    ax.fill_between(thc, np.append(g.q1, g.q1.iloc[0]),
                    np.append(g.q3, g.q3.iloc[0]), color=FRAME_COLORS[t], alpha=.10, lw=0)

ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=8, title="frame", bbox_to_anchor=(1.22, 1.0))
ax.set_title("Radius of peak intensity by angle\nmedian +/- IQR, pooled over nuclei")
fig.savefig(FIGDIR / "rose_pooled.png", bbox_inches="tight")

## 12 · Intensity by angle and radius

Rows are the focal tracks, columns timepoints. A ring that stays at fixed ρ
while the nucleus grows means the envelope is scaling with the nucleus.

In [ ]:
d = sweep[sweep.nucleus_id.isin(FOCAL)]
edges = np.linspace(0, 1, RHO_BINS + 1)
vmin, vmax = np.nanpercentile(d.intensity, [2, 98])

fig, axes = plt.subplots(len(FOCAL), len(FRAMES),
                         figsize=(2.6 * len(FRAMES), 2.4 * len(FOCAL)), squeeze=False)
im = None
for r, nid in enumerate(FOCAL):
    for k, t in enumerate(FRAMES):
        ax = axes[r][k]
        sub = d[(d.nucleus_id == nid) & (d.time_frame == t)].copy()
        if sub.empty:
            ax.set_facecolor("0.97"); ax.set_xticks([]); ax.set_yticks([])
        else:
            sub["rb"] = pd.cut(sub.rho_normalized, edges, labels=False, include_lowest=True)
            grid = sub.pivot_table(index="theta_deg", columns="rb", values="intensity",
                                   aggfunc="mean").reindex(columns=range(RHO_BINS))
            im = ax.imshow(grid.to_numpy(), aspect="auto", origin="lower", cmap="inferno",
                           vmin=vmin, vmax=vmax, extent=[0, 1, 0, 360])
            if r == len(FOCAL) - 1:
                ax.set_xlabel("rho", fontsize=8)
            else:
                ax.set_xticklabels([])
            if k == 0:
                ax.set_yticks([0, 180, 360])
            else:
                ax.set_yticklabels([])
        if r == 0:
            ax.set_title(f"t={t}", fontsize=9)
        if k == 0:
            ax.set_ylabel(f"{short(nid)}\ntheta (deg)", fontsize=8)

if im is not None:
    fig.colorbar(im, ax=axes, shrink=.6, label=f"{CHANNEL} intensity (a.u.)")
fig.suptitle("Intensity by angle and radius (rho, centre->wall) — focal tracks over time")
fig.savefig(FIGDIR / "angle_distance.png", bbox_inches="tight")

## 13 · Membrane asymmetry

`R` is the intensity-weighted resultant after a per-shell baseline subtraction.
`R_geom` is the resultant of sample counts alone — what an isotropic nucleus
would show purely from droplet-wall clipping. **R must beat R_geom.**

Shells are `ASYM_SHELLS` on `ASYM_RADIUS` (set in §1). Every subsection below
uses the same shells, so the R values in 13a–13e are the same numbers.

In [ ]:
asym = asymmetry(sweep, shells=ASYM_SHELLS, radius_col=ASYM_RADIUS)

SHELL_LABELS = [shell_label(lo, hi) for lo, hi in ASYM_SHELLS]
SHELL_COL    = dict(zip(SHELL_LABELS, SHELL_COLORS))
SHELL        = SHELL_LABELS[RING_SHELL_INDEX]                # the shell holding the ring
shells       = [s for s in SHELL_LABELS if s in set(asym.shell)]
RADIUS_LABEL = {"rho_wall": "rho_wall, surface->wall",
                "rho_normalized": "rho, centre->wall"}[ASYM_RADIUS]

empty = [s for s in SHELL_LABELS if s not in shells]
if empty:
    print("!! shells with no samples:", empty)
print(asym.shape, "| ring shell:", SHELL)
display(asym.groupby("shell")[["R", "R_geom", "n_eff"]].median().reindex(shells).round(3))

### 13a · Direction of the bright side over time

Grey bars are the population; coloured arrows are the focal tracks, so you can
see whether an individual nucleus holds its direction while the population stays
isotropic. Black line is the population resultant.

In [ ]:
NB = 24
d = asym[asym.shell == SHELL]
pop = population_direction(d).set_index("time_frame")
edges = np.linspace(0, 2 * np.pi, NB + 1)

fig, axes = plt.subplots(1, len(FRAMES), figsize=(3.1 * len(FRAMES), 4.0),
                         subplot_kw={"projection": "polar"}, squeeze=False)
for k, t in enumerate(FRAMES):
    ax = axes[0][k]; orient_polar(ax)
    phi = d.loc[d.time_frame == t, "phi"].dropna().to_numpy() % (2 * np.pi)
    counts, _ = np.histogram(phi, bins=edges)
    ax.bar(edges[:-1], counts, width=np.diff(edges), align="edge",
           color="0.82", edgecolor="w", lw=.4)
    hmax = counts.max() or 1
    if t in pop.index and np.isfinite(pop.loc[t, "R_pop"]):
        ax.plot([pop.loc[t, "mean_phi"]] * 2, [0, pop.loc[t, "R_pop"] * hmax],
                color="k", lw=1.8)
        ax.set_title(f"t={t}  n={phi.size}\nR={pop.loc[t,'R_pop']:.2f}, "
                     f"p={pop.loc[t,'p']:.3g}", fontsize=8)
    for nid in FOCAL:
        row = d[(d.nucleus_id == nid) & (d.time_frame == t)]
        if row.empty or not np.isfinite(row.phi.iloc[0]):
            continue
        ax.annotate("", xy=(row.phi.iloc[0], row.R.iloc[0] * hmax), xytext=(0, 0),
                    arrowprops=dict(color=FC[nid], width=1.4, headwidth=6, alpha=.9))
    ax.set_yticklabels([])

fig.legend(handles=[plt.Line2D([], [], color=FC[n], lw=2, label=short(n)) for n in FOCAL],
           loc="lower center", ncol=len(FOCAL), frameon=False, fontsize=8)
fig.suptitle(f"Direction of the bright side over time — shell {SHELL} ({RADIUS_LABEL})\n"
             "grey = population, coloured arrows = focal tracks, black = population resultant")
fig.tight_layout(rect=[0, .07, 1, .86])
fig.savefig(FIGDIR / "direction_windrose.png", bbox_inches="tight")

### 13b · Asymmetry against time

In [ ]:
d = (asym[asym.shell == SHELL]
     .merge(df_all[["nucleus_id", "time_frame", "time_min"]],
            on=["nucleus_id", "time_frame"], how="left"))

fig, (a, b, c) = plt.subplots(1, 3, figsize=(15, 4.4))

for _, g in d.groupby("nucleus_id"):
    g = g.sort_values("time_min")
    a.plot(g.time_min, g.R, color="0.82", lw=.5, alpha=.7)
plot_population(a, d, "R", points=False, label="population median +/- IQR")
plot_tracks(a, d, "R", FC)
a.set_xlabel("true acquisition time (min)"); a.set_ylabel("asymmetry score R")
a.set_title("(a) Asymmetry vs time"); a.legend(frameon=False, fontsize=8)

edges, labels = time_bins(d.time_min)
dd = d.assign(bin=pd.cut(d.time_min, edges, labels=labels,
                         include_lowest=True)).dropna(subset=["bin"])
order = [x for x in labels if (dd["bin"] == x).any()]
bp = boxplot(b, [dd.loc[dd["bin"] == x, "R"].to_numpy() for x in order], order,
             patch_artist=True, showfliers=False, medianprops=dict(color="k"))
for box in bp["boxes"]:
    box.set(facecolor="#f4c6a8", alpha=.9, edgecolor="#8a5a3b")
b.set_xlabel(f"time bin (min, width {int(BIN_MIN)})"); b.set_ylabel("asymmetry score R")
b.set_title("(b) Distribution per time bin")
plt.setp(b.get_xticklabels(), rotation=45, ha="right")

for nid in FOCAL:
    g = d[d.nucleus_id == nid].sort_values("time_frame")
    if len(g) < 2:
        continue
    dphi = np.degrees(np.angle(np.exp(1j * np.diff(g.phi.to_numpy()))))
    c.plot(g.time_frame.to_numpy()[1:], np.abs(dphi), "-o", ms=5, color=FC[nid],
           label=short(nid))
c.axhline(90, color="k", ls="--", lw=.8, label="90 deg = uncorrelated")
c.set_xlabel("time frame"); c.set_ylabel("|change in phi| from previous frame (deg)")
c.set_ylim(0, 180); c.set_yticks([0, 45, 90, 135, 180])
c.set_title("(c) Does a track hold its direction?"); c.legend(frameon=False, fontsize=8)

fig.suptitle(f"Perinuclear membrane asymmetry — shell {SHELL} ({RADIUS_LABEL})")
fig.tight_layout(rect=[0, 0, 1, .93])
fig.savefig(FIGDIR / "asymmetry_timecourse.png", bbox_inches="tight")

### 13c · By shell over time, against the geometry floor

In [ ]:
fig, (a, b, c) = plt.subplots(1, 3, figsize=(15, 4.4))
for s in shells:
    g = asym[asym.shell == s].groupby("time_frame")
    m = g.R.median()
    a.plot(m.index, m, "-o", color=SHELL_COL[s], ms=4, label=s)
    a.fill_between(m.index, g.R.quantile(.25), g.R.quantile(.75),
                   color=SHELL_COL[s], alpha=.15, lw=0)
    b.plot(m.index, m, "-o", color=SHELL_COL[s], ms=4, label=s)
    b.plot(m.index, g.R_geom.median(), ":^", color=SHELL_COL[s], ms=4)

a.axhline(np.sqrt(-np.log(.05) / max(int(asym.n_eff.median()), 1)), ls="--",
          color="k", lw=1, label="single-nucleus noise floor (p=0.05)")
a.set_xlabel("time frame"); a.set_ylabel("asymmetry score R")
a.set_title("(a) Asymmetry by shell over time"); a.legend(frameon=False, fontsize=7)
b.set_xlabel("time frame"); b.set_ylabel("R")
b.set_title("(b) Signal (solid) vs geometry floor (dotted)")

pop = population_direction(asym)
for s in shells:
    g = pop[pop.shell == s]
    c.plot(g.time_frame, g.R_pop, "-o", color=SHELL_COL[s], ms=4, label=s)
    c.plot(g.time_frame, np.sqrt(-np.log(.05) / g.n.clip(lower=1)), "--",
           color=SHELL_COL[s], lw=.8)
c.set_xlabel("time frame"); c.set_ylabel("population resultant of phi")
c.set_title("(c) Do nuclei agree on a lab direction?"); c.legend(frameon=False, fontsize=7)

fig.suptitle(f"Normalised asymmetry score by radial shell ({RADIUS_LABEL})")
fig.tight_layout(rect=[0, 0, 1, .93])
fig.savefig(FIGDIR / "asymmetry_by_shell.png", bbox_inches="tight")

### 13d · Aggregate, direction discarded

In [ ]:
NB = 36
fig, (a, b, c) = plt.subplots(1, 3, figsize=(15, 4.4))

data = [asym.loc[asym.shell == s, "R"].dropna().to_numpy() for s in shells]
parts = a.violinplot(data, showmedians=True, showextrema=True)
for pc, s in zip(parts["bodies"], shells):
    pc.set_facecolor(SHELL_COL[s]); pc.set_alpha(.55)
for i, s in enumerate(shells, start=1):
    a.plot([i - .35, i + .35], [asym.loc[asym.shell == s, "R_geom"].median()] * 2,
           ":", color="#c1121f", lw=1.6)
a.set_xticks(range(1, len(shells) + 1)); a.set_xticklabels(shells, fontsize=7)
a.set_xlabel(f"shell ({RADIUS_LABEL})"); a.set_ylabel("asymmetry score R")
a.plot([], [], ":", color="#c1121f", label="median R_geom")
a.set_title("(a) Score distribution, all frames"); a.legend(frameon=False, fontsize=8)

for s in shells:
    g = asym[asym.shell == s]
    for col, ls, lab in (("R", "-", s), ("R_geom", ":", None)):
        v = np.sort(g[col].dropna().to_numpy())
        if v.size:
            b.plot(v, np.arange(1, v.size + 1) / v.size, ls, color=SHELL_COL[s], lw=1.3, label=lab)
b.set_xlabel("asymmetry score R"); b.set_ylabel("cumulative fraction")
b.set_title("(b) ECDF (dotted = R_geom)"); b.legend(frameon=False, fontsize=7)

# Same shells and radius as asymmetry(), so each profile is aligned on its own R/phi.
phi_lookup = asym.set_index(["nucleus_id", "time_frame", "shell"])["phi"]
centres = np.linspace(-180, 180, NB, endpoint=False) + 180 / NB
for (lo, hi), s in zip(ASYM_SHELLS, SHELL_LABELS):
    if s not in shells:
        continue
    sub_all = sweep[(sweep[ASYM_RADIUS] > lo) & (sweep[ASYM_RADIUS] <= hi)]
    stack = []
    for (nid, t), g in sub_all.groupby(["nucleus_id", "time_frame"], sort=False):
        phi = phi_lookup.get((nid, t, s), np.nan)
        if not np.isfinite(phi):
            continue
        _, mean, _ = angular_profile(g, NB)
        if not np.isfinite(mean).any() or np.nanmean(mean) <= 0:
            continue
        shift = int(round(np.degrees(phi) / (360 / NB)))
        stack.append(np.roll(mean / np.nanmean(mean), -shift + NB // 2))
    if stack:
        arr = np.vstack(stack)
        med = np.nanmedian(arr, axis=0)
        se = np.nanstd(arr, axis=0) / max(np.sqrt(arr.shape[0]), 1)
        c.plot(centres, med, color=SHELL_COL[s], lw=1.4, label=f"{s} (n={arr.shape[0]})")
        c.fill_between(centres, med - se, med + se, color=SHELL_COL[s], alpha=.2, lw=0)
c.axhline(1.0, color="k", lw=.7)
c.set_xlabel("angle from each nucleus's own asymmetry axis (deg)")
c.set_ylabel("intensity / shell mean"); c.set_xticks([-180, -90, 0, 90, 180])
c.set_title("(c) Alignment-averaged profile"); c.legend(frameon=False, fontsize=7)

fig.suptitle("Aggregate membrane asymmetry — direction discarded")
fig.tight_layout(rect=[0, 0, 1, .93])
fig.savefig(FIGDIR / "asymmetry_aggregate.png", bbox_inches="tight")

### 13e · Shell x frame rose grid, one figure per focal track

Wedge radius is sqrt(intensity above baseline), so wedge **area** is proportional
to intensity. Grey wedges are angles with no sample — almost always droplet-wall
clipping. Red arrow is the resultant.

In [ ]:
NB = 36
edges = np.linspace(0, 2 * np.pi, NB + 1)

for nid in FOCAL:
    d = sweep[sweep.nucleus_id == nid]
    fig, axes = plt.subplots(len(ASYM_SHELLS), len(FRAMES),
                             figsize=(2.2 * len(FRAMES), 2.4 * len(ASYM_SHELLS)),
                             subplot_kw={"projection": "polar"}, squeeze=False)
    for r, ((lo, hi), col) in enumerate(zip(ASYM_SHELLS, SHELL_COLORS)):
        for k, t in enumerate(FRAMES):
            ax = axes[r][k]; orient_polar(ax)
            ax.set_xticklabels([]); ax.set_yticklabels([])
            sub = d[(d.time_frame == t) & (d[ASYM_RADIUS] > lo) & (d[ASYM_RADIUS] <= hi)]
            if sub.empty:
                ax.set_facecolor("0.97")
            else:
                th, mean, count = angular_profile(sub, NB)
                base = np.nanpercentile(mean[np.isfinite(mean)], 10)
                w = np.clip(np.nan_to_num(mean - base), 0, None)
                h = np.sqrt(w); hmax = h.max() or 1
                miss = count == 0
                if miss.any():
                    ax.bar(edges[:-1][miss], np.full(miss.sum(), hmax),
                           width=np.diff(edges)[miss], align="edge",
                           color="0.88", zorder=0)
                ax.bar(edges[:-1], h, width=np.diff(edges), align="edge",
                       color=col, edgecolor="w", lw=.3)
                R, phi = resultant(th, w)
                if np.isfinite(R):
                    ax.annotate("", xy=(phi, R * hmax), xytext=(0, 0),
                                arrowprops=dict(color="#c1121f", width=1.2, headwidth=6))
                    ax.set_title(f"R={R:.2f}", fontsize=7, color="0.3")
            if k == 0:
                ax.set_ylabel(shell_label(lo, hi), fontsize=7)
            if r == 0:
                ax.text(.5, 1.30, f"t = {t}", transform=ax.transAxes,
                        ha="center", fontsize=9)
    fig.suptitle(f"Intensity by angle and shell ({RADIUS_LABEL}) over time — {short(nid)}\n"
                 "wedge area prop. to intensity above baseline · red = resultant · "
                 "grey = no sample", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, .90])
    fig.savefig(FIGDIR / f"shell_rose_{short(nid)}.png", bbox_inches="tight")
    plt.show()

## 14 · Intensity surfaces over time

Rows are the focal tracks, columns timepoints. ρ and θ are mapped to Cartesian,
so each surface sits over the nucleus's own footprint with the nucleus at the
centre and Z as intensity. `ZLIM` is set here from the focal tracks and reused in
14c, so focal and pooled surfaces share one scale.

In [ ]:
ZLIM = tuple(np.nanpercentile(sweep.loc[sweep.nucleus_id.isin(FOCAL), "intensity"], [1, 99]))
zmin, zmax = ZLIM

fig = plt.figure(figsize=(3.0 * len(FRAMES), 3.0 * len(FOCAL)))
for r, nid in enumerate(FOCAL):
    for k, t in enumerate(FRAMES):
        ax = fig.add_subplot(len(FOCAL), len(FRAMES), r * len(FRAMES) + k + 1,
                             projection="3d")
        one = sweep[(sweep.nucleus_id == nid) & (sweep.time_frame == t)]
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        if one.empty:
            ax.set_axis_off()
        else:
            THETA, RHO, Z = sweep_grid(one)
            X, Y = polar_xy(THETA, RHO)
            ax.plot_surface(X, Y, fill_gaps(Z), cmap="inferno", linewidth=0,
                            antialiased=True, vmin=zmin, vmax=zmax)
            ax.set_zlim(zmin, zmax)
            ax.view_init(elev=45, azim=-58)
        if r == 0:
            ax.set_title(f"t = {t}", fontsize=9)
        if k == 0:
            ax.text2D(-0.15, 0.5, short(nid), transform=ax.transAxes,
                      rotation=90, va="center", fontsize=9)

fig.suptitle(f"{CHANNEL} intensity surfaces — focal tracks over time "
             f"(shared Z scale {zmin:.0f}-{zmax:.0f})", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, .95])
fig.savefig(FIGDIR / "surface_focal_grid.png", bbox_inches="tight")

### 14b · One track, large enough to read the envelope structure

In [ ]:
NID, T = FOCAL[0], FRAMES[-1]
one = sweep[(sweep.nucleus_id == NID) & (sweep.time_frame == T)]
if one.empty:
    raise ValueError(f"{short(NID)} has no sweep samples at t={T} -- pick another NID/T")

THETA, RHO, Z = sweep_grid(one)
X, Y = polar_xy(THETA, RHO)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(X, Y, fill_gaps(Z), cmap="inferno", linewidth=0, antialiased=True,
                       rstride=1, cstride=1)
ax.set_xlabel("x (rho)"); ax.set_ylabel("y (rho)"); ax.set_zlabel("intensity")
ax.view_init(elev=42, azim=-58)
fig.colorbar(surf, shrink=.55, label=f"{CHANNEL} intensity (a.u.)")
ax.set_title(f"{CHANNEL} intensity surface — {short(NID)}, t={T}")
fig.savefig(FIGDIR / "surface_polar.png", bbox_inches="tight")

### 14c · Pooled across nuclei, over time

Drop the focal filter and the pivot's mean pools everything — shows whether a
feature is systematic or belongs to one nucleus. Same `ZLIM` as §14.

In [ ]:
zmin, zmax = ZLIM
fig = plt.figure(figsize=(3.2 * len(FRAMES), 3.6))
for k, t in enumerate(FRAMES):
    ax = fig.add_subplot(1, len(FRAMES), k + 1, projection="3d")
    allnuc = sweep[sweep.time_frame == t]
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    if allnuc.empty:
        ax.set_axis_off()
        continue
    THETA_a, RHO_a, Z_a = sweep_grid(allnuc)
    X_a, Y_a = polar_xy(THETA_a, RHO_a)
    ax.plot_surface(X_a, Y_a, fill_gaps(Z_a), cmap="viridis", linewidth=0,
                    antialiased=True, vmin=zmin, vmax=zmax)
    ax.set_zlim(zmin, zmax)
    ax.view_init(elev=45, azim=-58)
    ax.set_title(f"t = {t}  (n={allnuc.nucleus_id.nunique()})", fontsize=9)

fig.suptitle("Pooled intensity surface over time — shared Z scale", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, .90])
fig.savefig(FIGDIR / "surface_pooled.png", bbox_inches="tight")

## 15 · Envelope band, surface-referenced

**What this can and cannot show.** The sweep runs with `exclude_nucleus=True`,
so the export contains nothing inside the nucleus mask — the envelope itself and
the interior are absent. What is available is the first shell *outside* the mask,
where NE-associated membrane signal sits. `rho_wall ≤ BAND` is that band.

### 15a · Unwrapped band, θ × time

In [ ]:
NB_THETA = 90
env = sweep[sweep.rho_wall <= BAND]
print(f"envelope band (rho_wall <= {BAND}): {len(env):,} of {len(sweep):,} samples")

edges = np.linspace(0, 360, NB_THETA + 1)
vmin, vmax = np.nanpercentile(env.intensity, [2, 98])

fig, axes = plt.subplots(1, len(FOCAL), figsize=(3.6 * len(FOCAL), 4.6),
                         squeeze=False, sharey=True)
im = None
for ax, nid in zip(axes[0], FOCAL):
    d = env[env.nucleus_id == nid].copy()
    d["tb"] = pd.cut(d.theta_deg % 360, edges, labels=False, include_lowest=True)
    grid = (d.pivot_table(index="time_frame", columns="tb", values="intensity", aggfunc="mean")
            .reindex(index=FRAMES, columns=range(NB_THETA)))
    # Rows are drawn by position, so the extent is in row units -- correct even when FRAMES skips a frame.
    im = ax.imshow(grid.to_numpy(), aspect="auto", origin="lower", cmap="inferno",
                   vmin=vmin, vmax=vmax, extent=[0, 360, -.5, len(FRAMES) - .5])
    ax.set_title(short(nid), fontsize=10)
    ax.set_xlabel("theta (deg)")
    ax.set_xticks([0, 90, 180, 270, 360])
    ax.set_yticks(range(len(FRAMES)), [str(t) for t in FRAMES])
axes[0][0].set_ylabel("time frame")
if im is not None:
    fig.colorbar(im, ax=axes, shrink=.75, label=f"{CHANNEL} intensity, envelope band")
fig.suptitle(f"Unwrapped envelope band (0 to {BAND} of the surface-to-wall gap)\n"
             "a lobe holding its theta across rows is stable polarity; "
             "blank rows are undetected frames", y=1.06)
fig.savefig(FIGDIR / "envelope_unwrapped_kymograph.png", bbox_inches="tight")

### 15b · Depth-resolved, one figure per focal track

θ × `rho_wall` out to three band-widths; dashed line marks `BAND`. Panels go
sparse as the nucleus grows and the surface-to-wall gap closes — see
**A · Diagnostics** for the check.

In [ ]:
NB_THETA, NB_RHO = 72, 30
RHO_MAX = BAND * 3
NCOL = 4
te  = np.linspace(0, 360, NB_THETA + 1)
re_ = np.linspace(0, RHO_MAX, NB_RHO + 1)

for nid in FOCAL:
    d = sweep[(sweep.nucleus_id == nid) & (sweep.rho_wall <= RHO_MAX)].copy()
    if d.empty:
        continue
    d["tb"] = pd.cut(d.theta_deg % 360, te, labels=False, include_lowest=True)
    d["rb"] = pd.cut(d.rho_wall, re_, labels=False, include_lowest=True)
    vmin, vmax = np.nanpercentile(d.intensity, [2, 98])

    nrow = int(np.ceil(len(FRAMES) / NCOL))
    fig, axes = plt.subplots(nrow, NCOL, figsize=(3.6 * NCOL, 3.0 * nrow),
                             squeeze=False, sharex=True, sharey=True)
    im = None
    for k, t in enumerate(FRAMES):
        ax = axes[k // NCOL][k % NCOL]
        sub = d[d.time_frame == t]
        if sub.empty:
            ax.set_facecolor("0.97")
            ax.text(.5, .5, "no detection", transform=ax.transAxes,
                    ha="center", va="center", color="0.6", fontsize=9)
        else:
            grid = (sub.pivot_table(index="tb", columns="rb", values="intensity", aggfunc="mean")
                    .reindex(index=range(NB_THETA), columns=range(NB_RHO)))
            im = ax.imshow(grid.to_numpy(), aspect="auto", origin="lower",
                           cmap="inferno", vmin=vmin, vmax=vmax,
                           extent=[0, RHO_MAX, 0, 360])
            ax.axvline(BAND, color="w", lw=.8, ls="--")
        ax.set_title(f"t = {t}", fontsize=10)
        ax.set_yticks([0, 90, 180, 270, 360])
        if k // NCOL == nrow - 1:
            ax.set_xlabel("rho_wall (0 = mask surface)")
        if k % NCOL == 0:
            ax.set_ylabel("theta (deg)")
    for k in range(len(FRAMES), nrow * NCOL):
        axes[k // NCOL][k % NCOL].axis("off")
    if im is not None:
        fig.colorbar(im, ax=axes, shrink=.7, label=f"{CHANNEL} intensity")
    fig.suptitle(f"Envelope shell, depth-resolved — {short(nid)}   "
                 f"(dashed = BAND at {BAND})", y=1.00)
    fig.savefig(FIGDIR / f"envelope_shell_{short(nid)}.png", bbox_inches="tight")
    plt.show()

## 16 · Figures written

In [ ]:
for p in sorted(FIGDIR.glob("*.png")):
    print(f"{p.stat().st_size/1e3:8.0f} kB  {p.name}")

## A · Diagnostics (one-off)

Informational checks kept for reference. Nothing above depends on them, and they
write no figures.

In [ ]:
# DIAGNOSTIC — one-off, informational. Not a fix.
# Is the sparsity in the depth-resolved envelope plot (§15b) a binning problem?
# Plots samples per (theta, rho_wall) cell, and the surface radius over time, for the first
# focal track on the same grid as §15b. If sample count collapses while the nucleus grows,
# the bin grid is the issue.
NB_THETA, NB_RHO = 72, 30
RHO_MAX = BAND * 3
NID = FOCAL[0]

d = sweep[(sweep.nucleus_id == NID) & (sweep.rho_wall <= RHO_MAX)].copy()
te  = np.linspace(0, 360, NB_THETA + 1)
re_ = np.linspace(0, RHO_MAX, NB_RHO + 1)
d["tb"] = pd.cut(d.theta_deg % 360, te, labels=False, include_lowest=True)
d["rb"] = pd.cut(d.rho_wall, re_, labels=False, include_lowest=True)

rows = []
for t in FRAMES:
    sub = d[d.time_frame == t]
    if sub.empty:
        rows.append(dict(t=t, n_samples=0, empty_cells=np.nan,
                         median_per_cell=np.nan, rho_surface=np.nan, gap=np.nan))
        continue
    cnt = (sub.groupby(["tb", "rb"]).size()
           .reindex(pd.MultiIndex.from_product([range(NB_THETA), range(NB_RHO)]))
           .fillna(0))
    rs = sweep.loc[(sweep.nucleus_id == NID) & (sweep.time_frame == t), "rho_surface"].median()
    rows.append(dict(t=t, n_samples=len(sub),
                     empty_cells=round(100 * (cnt == 0).mean(), 1),
                     median_per_cell=cnt.median(),
                     rho_surface=round(rs, 3), gap=round(1 - rs, 3)))
diag = pd.DataFrame(rows)
display(diag)
print("empty_cells = % of (theta, rho) bins with zero samples")
print("gap = 1 - rho_surface, the fraction of each ray outside the nucleus")

fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot(diag.t, diag.empty_cells, "-o", color="#c1121f")
a.set_xlabel("time frame"); a.set_ylabel("% empty bins")
a.set_title("(a) Coverage of the bin grid")
b.plot(diag.t, diag.gap, "-o", color="#0a6f8a", label="surface-to-wall gap")
b2 = b.twinx()
b2.plot(diag.t, diag.n_samples, "-s", color="#5b3a91", label="samples in band")
b.set_xlabel("time frame"); b.set_ylabel("gap (fraction of ray)", color="#0a6f8a")
b2.set_ylabel("samples", color="#5b3a91")
b2.spines["right"].set_visible(True)
b.set_title("(b) The gap closes as the nucleus grows")
fig.suptitle(f"Why the depth-resolved panels go sparse — {short(NID)}")
fig.tight_layout()